# UTS DL – training di Colab (Task 2 & 3)
**Cara pakai:** Runtime → Change runtime type → **T4 GPU**, lalu **Runtime → Run all**.

Semua output (checkpoint, grafik, tabel, CSV) disimpan ke Google Drive (`OUT_DIR`). Jika sesi terputus, cukup **Run all** lagi: (model, fold) yang sudah selesai dilewati dan training yang terputus dilanjutkan dari checkpoint epoch terakhir (hasil identik dengan tanpa putus).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# --- Sesuaikan bila perlu ---
REPO_URL = 'https://github.com/roplyhmn/UTS-Deep-Learning.git'
BRANCH   = 'task2/rul'                                  # branch yang sedang dikerjakan
ZIP_PATH = '/content/drive/MyDrive/data_uts_dl.zip'     # zip resmi dari dosen
OUT_DIR  = '/content/drive/MyDrive/uts_dl_outputs'      # hasil di Drive (aman jika sesi putus)
DATA_DIR = '/content/data'
RUN_TASK2 = True
RUN_W100  = True                                        # eksperimen window 100 (tidak menimpa CSV final)
RUN_TASK3 = False                                       # diaktifkan setelah Task 3 siap

In [ ]:
import os
if not os.path.exists('/content/UTS-Deep-Learning'):
    !git clone -q -b $BRANCH $REPO_URL /content/UTS-Deep-Learning
else:
    !git -C /content/UTS-Deep-Learning fetch -q origin $BRANCH && git -C /content/UTS-Deep-Learning checkout -q $BRANCH && git -C /content/UTS-Deep-Learning reset -q --hard origin/$BRANCH
%cd /content/UTS-Deep-Learning
!git log --oneline -1
!pip install -q -r kode/requirements.txt

In [ ]:
# Ekstrak data (zip bisa berisi folder 'data/' atau berkas langsung - keduanya ditangani)
import zipfile, glob, shutil
if not os.path.exists(f'{DATA_DIR}/rul_train.csv.gz'):
    os.makedirs(DATA_DIR, exist_ok=True)
    zipfile.ZipFile(ZIP_PATH).extractall('/content/_unzip')
    for f in glob.glob('/content/_unzip/**/*.*', recursive=True):
        shutil.move(f, os.path.join(DATA_DIR, os.path.basename(f)))
print(sorted(os.listdir(DATA_DIR)))

In [ ]:
# Cek GPU + catat versi library (untuk requirements.txt dan laporan)
import torch, json
assert torch.cuda.is_available(), 'GPU tidak aktif: Runtime -> Change runtime type -> T4 GPU'
os.makedirs(OUT_DIR, exist_ok=True)
!python -c "import json; from kode.common.utils import env_info; i=env_info(); print(json.dumps(i, indent=1)); json.dump(i, open('$OUT_DIR/env_info.json','w'), indent=1)"

In [ ]:
# Uji anti-kebocoran Task 2 (~10 detik)
if RUN_TASK2:
    !python -m kode.task2.test_leakage --data_dir $DATA_DIR

In [ ]:
# Uji cepat seluruh pipeline Task 2 (subset kecil, < 1 menit)
if RUN_TASK2:
    !python -m kode.task2.main --debug --data_dir $DATA_DIR --out_dir $OUT_DIR

In [ ]:
# Task 2a-c: grafik EDA & window (data penuh, ~1 menit)
if RUN_TASK2:
    !python -m kode.task2.eda --data_dir $DATA_DIR --out_dir $OUT_DIR
    !python -m kode.task2.windows --data_dir $DATA_DIR --out_dir $OUT_DIR

In [ ]:
# Task 2d: TRAINING PENUH - baseline + LSTM + TCN, 5 fold. Resume otomatis.
if RUN_TASK2:
    !python -m kode.task2.main --data_dir $DATA_DIR --out_dir $OUT_DIR --pred_dir $OUT_DIR

In [ ]:
# Eksperimen: window W=100 (histori lebih panjang). Hasil di task2/models_W100, CSV final tidak ditimpa
if RUN_TASK2 and RUN_W100:
    !python -m kode.task2.main --W 100 --no-write_pred --data_dir $DATA_DIR --out_dir $OUT_DIR

In [ ]:
# Ringkasan hasil Task 2
import pandas as pd
from IPython.display import Image, display
if RUN_TASK2:
    m = f'{OUT_DIR}/task2/models'
    display(pd.read_csv(f'{m}/tabel_perbandingan_model.csv'))
    if RUN_W100 and os.path.exists(f'{m}_W100/tabel_perbandingan_model.csv'):
        print('W=100:'); display(pd.read_csv(f'{m}_W100/tabel_perbandingan_model.csv'))
    display(pd.read_csv(f'{m}/tabel_per_rentang_rul.csv').pivot(index='rentang_rul', columns='model', values='rmse'))
    display(pd.read_csv(f'{m}/tabel_per_mode.csv'))
    for g in ['10_rmse_per_rentang', '11_prediksi_vs_aktual', '12_kurva_training']:
        display(Image(f'{m}/{g}.png', width=800))

In [ ]:
# Validasi format CSV prediksi
!python -m kode.validasi_csv --data_dir $DATA_DIR --pred_dir $OUT_DIR

Setelah selesai: unduh folder `OUT_DIR/task2/` (tanpa `checkpoints/`) atau beri tahu Claude agar hasilnya dimasukkan ke repo.